In [1]:
import requests
import pandas as pd
import time

API_KEY = "c96ddcf8e9d62c9de8732d7f42e97098"
BASE_URL = "https://api.themoviedb.org/3"

In [2]:
movie_ids = []

for page in range(1, 126):  # 125 halaman x 20 = 2500 ID (buffer krn nanti banyak yg budget/revenue-nya 0)
    params = {
        "api_key": API_KEY,
        "sort_by": "popularity.desc",
        "vote_count.gte": 50,
        "page": page
    }
    resp = requests.get(f"{BASE_URL}/discover/movie", params=params)
    data = resp.json()
    for movie in data.get("results", []):
        movie_ids.append(movie["id"])
    time.sleep(0.3)

movie_ids = list(set(movie_ids))  # hapus duplikat
print("Total ID unik terkumpul:", len(movie_ids))

Total ID unik terkumpul: 1981


In [3]:
records = []

for i, mid in enumerate(movie_ids):
    resp = requests.get(f"{BASE_URL}/movie/{mid}", params={"api_key": API_KEY})
    if resp.status_code == 200:
        d = resp.json()
        records.append({
            "id": d.get("id"),
            "title": d.get("title"),
            "budget": d.get("budget"),
            "revenue": d.get("revenue"),
            "runtime": d.get("runtime"),
            "popularity": d.get("popularity"),
            "vote_average": d.get("vote_average"),
            "vote_count": d.get("vote_count"),
            "release_date": d.get("release_date"),
            "genres": ", ".join(g["name"] for g in d.get("genres", []))
        })
    time.sleep(0.3)
    if i % 200 == 0:
        print(f"{i}/{len(movie_ids)} film diproses...")

df = pd.DataFrame(records)
print(df.shape)
df.head()

0/1981 film diproses...
200/1981 film diproses...
400/1981 film diproses...
600/1981 film diproses...
800/1981 film diproses...
1000/1981 film diproses...
1200/1981 film diproses...
1400/1981 film diproses...
1600/1981 film diproses...
1800/1981 film diproses...
(1981, 10)


,id,title,budget,revenue,runtime,popularity,vote_average,vote_count,release_date,genres
0,1171462,Golden Kamuy,0,19077748,129,6.4613,7.037,95,2024-01-19,"Action, Adventure, Comedy"
1,8204,The Spiderwick Chronicles,90000000,164200000,95,7.1093,6.707,2896,2008-02-14,"Family, Adventure, Fantasy, Drama"
2,14,American Beauty,15000000,356296601,122,10.9627,7.999,13153,1999-09-15,Drama
3,18,The Fifth Element,90000000,263920180,126,19.8421,7.586,11891,1997-05-02,"Science Fiction, Action, Adventure"
4,19,Metropolis,5300000,1350322,148,5.4421,8.082,3133,1927-01-10,"Drama, Science Fiction"


In [4]:
df_clean = df[
    (df["budget"] > 0) &
    (df["revenue"] > 0) &
    (df["runtime"] > 0)
].drop_duplicates(subset="id").reset_index(drop=True)

print("Jumlah baris setelah cleaning:", df_clean.shape[0])
df_clean.to_csv("movies_dataset.csv", index=False)

Jumlah baris setelah cleaning: 1589


In [10]:
from google.colab import files
files.download("movies_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>